# 1.0 交通行业财务数据——因子分析

# 1.1 概述
近年来，关于交通运输业上市公司的股票投资备受关注。投资者为了获得更多的收益，需要通过收集公司的财务数据，对公司进行各方面的分析。
本次案例需要对交通运输业上市公司的财务数据进行因子分析，并通过因子分析得到衡量这些上市公司经营好坏的指标，进而掌握这些上市公司的经营状况。

字段名称	字段解释
X1	基本每股收益(元)
X2	每股净资产(元)
X3	净资产收益率
X4	净利润率
X5 	总资产报酬率
X6	存货周转率
X7	固定资产周转率
X8	总资产周转率


## 1.2 导入相关库及数据

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing
from factor_analyzer import FactorAnalyzer
data = pd.read_table('交通数据.txt')
data.set_index('上市公司', inplace= True)    # 将上市公司设为索引
data

,X1,X2,X3,X4,X5,X6,X7,X8
上市公司,,,,,,,,
西部创业,0.0600,2.7591,2.1900,13.19,1.72,5.90,0.19,0.13
铁龙物流,0.2530,4.0325,6.4380,2.83,4.05,3.63,4.22,1.43
大秦铁路,0.9000,6.6862,13.5000,23.99,10.63,25.57,0.77,0.44
广深铁路,0.1400,4.0495,3.5900,5.54,3.04,50.13,0.77,0.55
渤海轮渡,0.7500,6.7324,11.6700,24.00,8.83,25.79,0.48,0.37
深圳机场,0.3225,5.4464,6.0500,19.92,5.17,438.82,0.44,0.26
中信海直,0.1500,4.8626,3.1500,7.17,1.73,3.22,0.42,0.24
白云机场,0.8200,7.2518,12.2400,23.60,7.93,53.26,0.87,0.34
上海机场,1.9100,13.0420,15.5300,45.69,13.52,206.53,0.86,0.30


## 1.2 数据标准化

In [3]:
# 标准化
x_scale = preprocessing.scale(data)   # 标准化、
x_scale[0] # 显示第一个数据


array([-1.12131357, -1.10165742, -1.29905301, -0.31671996, -0.95683513,
       -0.29740377, -0.83687094, -1.02173589])

## 2.0 确定因子数目

In [4]:
# 计算相关系数矩阵
covX = np.around(np.corrcoef(x_scale.T), decimals = 3)
# 系数相关矩阵特征求解
featValue, featVec= np.linalg.eig(covX.T)    # featValue为特征根，featVec为特征向量
# 计算贡献率
# 特征根排序输出
featValue = sorted(featValue)[::-1]
# 信息贡献度
gx = featValue / np.sum(featValue)
#累计贡献度
lg = np.cumsum(gx)
lg

array([0.4938552 , 0.71129633, 0.84140138, 0.90857376, 0.96131223,
       0.98497114, 0.99584533, 1.        ])

## 3.0 计算载荷矩阵

### 3.1 主成分法

In [5]:
# 主成分法
fa_c = FactorAnalyzer(n_factors=3, rotation= 'varimax',method= 'principal')   # rotation参数为是否旋转，值varimax是使得方差最大的旋转
fa_c.fit(x_scale)
fa_c.get_factor_variance()[2]  # 输出累积贡献率

array([0.41832825, 0.63826147, 0.84136648])

### 3.2 主因子法

In [6]:
# 主因子法
fa_y = FactorAnalyzer(n_factors=3, rotation= 'varimax',method= 'minres')
fa_y.fit(x_scale)
fa_y.get_factor_variance()[2]

array([0.33771333, 0.55153909, 0.73881338])

### 3.3 极大似然法

In [7]:
# 极大似然法
fa_s = FactorAnalyzer(n_factors=3, rotation= 'varimax',method= 'ml')
fa_s.fit(x_scale)
fa_s.get_factor_variance()[2]

array([0.31900365, 0.55677071, 0.75005086])

### 3.4 输出载荷矩阵

In [8]:
# 输出载荷矩阵
print(pd.DataFrame(fa_c.loadings_, index= data.columns))

           0         1         2
X1  0.829465  0.057875  0.509404
X2  0.557740  0.109024  0.687048
X3  0.858548 -0.173595  0.052840
X4  0.809524 -0.368892 -0.011217
X5  0.927820  0.048310  0.235513
X6  0.046403 -0.095718  0.900408
X7  0.062032  0.892441 -0.120996
X8 -0.297078  0.877535  0.097750


## 4.0 计算因子得分

In [10]:
# 输出因子得分
sc1 = fa_c.transform(x_scale)    
print(sc1)

[[-1.10331451e+00 -1.17008495e+00 -4.34701216e-01]
 [-6.65473137e-01  2.23003949e+00 -1.46022107e-01]
 [ 1.01626488e+00 -8.34330424e-02 -1.11024770e-01]
 [-1.05465425e+00 -1.78078502e-01 -3.96544351e-02]
 [ 6.78927003e-01 -3.55956212e-01 -7.42907949e-02]
 [-3.85050975e-01 -7.98485690e-01  2.11980106e-01]
 [-1.09800373e+00 -7.88980493e-01 -6.59030281e-03]
 [ 6.96360158e-01 -2.98956493e-01  1.91983786e-02]
 [ 2.46668713e+00 -4.87072675e-02  9.27367143e-01]
 [-3.41695227e-01  4.17706082e-02  3.86591409e-02]
 [-3.97292792e-01 -3.12759934e-01 -2.86548352e-01]
 [-1.07598230e+00 -5.70176517e-01  1.64075099e-02]
 [ 2.05521926e+00  2.18419156e+00  5.26001823e-01]
 [-8.99744534e-02 -7.08468590e-01  4.74980206e+00]
 [-4.92912954e-01 -6.64194361e-02  2.86862877e-01]
 [-1.14584293e-01  3.04778223e-01 -7.07006486e-01]
 [ 1.07047191e+00 -5.56642297e-01 -1.14587726e+00]
 [ 7.71162013e-03  7.88536293e-01 -2.71317268e-01]
 [-4.67717941e-01 -3.41591117e-01 -2.37081162e-01]
 [-5.81777902e-02  1.10543384e+